<a href="https://colab.research.google.com/github/himanshusar123/-ClaimVision-AI/blob/main/Day16_GenerativeAI_Lab_Validated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 16 | Generative AI — Validated Multimodal Lab (BrightCart)
Improved version of the Day 16 workbook code. Every cell is **self-contained** (imports included), so you can re-run any cell on its own.

**Cells that need internet / large downloads** (2C text model, 3B Stable Diffusion, 4 gTTS) are wrapped so a failure prints a clear message instead of a traceback; the rest of the notebook keeps working.

Run top to bottom with Shift+Enter. Never paste customer data, passwords or API keys into any AI tool.

## Cell 1 — Environment check

In [1]:
import sys, platform
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

try:
    import torch
    print("PyTorch:", torch.__version__, "| GPU available:", torch.cuda.is_available())
except ImportError:
    print("PyTorch not installed yet (only needed for the optional text/image model cells).")

Python: 3.13.16
Platform: Linux-6.6.122+-x86_64-with-glibc2.39
PyTorch: 2.11.0+cpu | GPU available: False


## Part 2 — Text generation
### 2A — Structured prompt
Same prompt as the workbook; it now also checks that inputs are non-empty strings.

In [2]:
def build_prompt(customer_message, order_context):
    if not str(customer_message).strip() or not str(order_context).strip():
        raise ValueError("customer_message and order_context must not be empty")
    return f"""Role: BrightCart customer-support assistant

Task: Draft a helpful response to the customer.

Order context: {order_context}

Customer message: {customer_message}

Constraints:
- Be polite and concise.
- Do not invent refund approval, delivery dates, or policy terms.
- If information is missing, ask one clear question.

Output format: Reply: <draft>
"""

prompt = build_prompt(
    "My package has not arrived. Can you refund me?",
    "Order status: delayed; refund eligibility not checked",
)
print(prompt)
assert "Do not invent refund approval" in prompt

Role: BrightCart customer-support assistant

Task: Draft a helpful response to the customer.

Order context: Order status: delayed; refund eligibility not checked

Customer message: My package has not arrived. Can you refund me?

Constraints:
- Be polite and concise.
- Do not invent refund approval, delivery dates, or policy terms.
- If information is missing, ask one clear question.

Output format: Reply: <draft>



### 2B — Rule-based fallback (not an LLM)
**Fixes:** the original compared `status == "delayed"` exactly, so `"Delayed"`, `" delayed "` or `None` silently skipped the delay branch. Input is now normalised, and tests are included.

In [3]:
def brightcart_fallback(message, status):
    message_l = (message or "").lower()
    status_l = (status or "").strip().lower()
    if status_l == "delayed":
        return ("I'm sorry your order is delayed. I can help check the latest tracking "
                "details. Your refund eligibility has not yet been confirmed.")
    if "refund" in message_l:
        return ("I can help you check the refund policy. Please share your order status "
                "or contact support with your order number.")
    return ("Thanks for contacting BrightCart. Could you share your order number "
            "and a little more detail so I can help?")

reply = brightcart_fallback("My package has not arrived. Can you refund me?", "delayed")
print(reply)

# tests
assert brightcart_fallback("refund?", "Delayed") == reply          # case-insensitive
assert brightcart_fallback("refund?", "  delayed ") == reply       # whitespace
assert "refund policy" in brightcart_fallback("I want a refund", "shipped")
assert "order number" in brightcart_fallback("hello", None)         # None-safe
assert brightcart_fallback(None, None)                              # no crash
print("2B tests passed")

I'm sorry your order is delayed. I can help check the latest tracking details. Your refund eligibility has not yet been confirmed.
2B tests passed


### 2B+ — Guardrail check for unsupported promises *(new)*
The workbook says outputs must not invent refunds, dates or policy terms, but nothing tested it. This simple checker flags risky phrases. It is a **first-pass filter, not a guarantee** — human review is still required.

In [4]:
import re

RISKY_PATTERNS = {
    "refund promise": r"(refund (has been|is|will be) (approved|issued|processed)|we (will|'ll) refund|you (will|'ll) (get|receive) (a )?refund|i (will|'ll) refund)",
    "delivery-date promise": r"(will (arrive|be delivered) (on|by|tomorrow|today)|delivered (on|by) )",
    "invented policy": r"(\b30[- ]day\b|within \d+ (business )?days)",
}

def check_reply(text):
    """Return a list of risk labels found in a draft reply (empty list = nothing flagged)."""
    return [label for label, pat in RISKY_PATTERNS.items() if re.search(pat, text or "", re.I)]

assert check_reply(reply) == []
assert check_reply("Good news, your refund has been approved!") == ["refund promise"]
assert check_reply("It will arrive by Friday.") == ["delivery-date promise"]
assert "invented policy" in check_reply("Refunds are issued within 5 business days.")
print("Guardrail tests passed. Flags on safe reply:", check_reply(reply))

Guardrail tests passed. Flags on safe reply: []


### 2C — Optional: real text generation with DistilGPT-2
**Fixes:** a few-shot prompt (DistilGPT-2 is not instruction-tuned, so the long role/constraint prompt confuses it), fixed seed for repeatable output, `return_full_text=False`, pad token taken from the tokenizer instead of hard-coded `50256`, `top_p`/`repetition_penalty` for less rambling, graceful failure, and the guardrail check applied to the result.
Needs internet and about 1 GB of free RAM.

In [5]:
!pip -q install transformers torch

In [6]:
few_shot_prompt = (
    "Customer: Where is my order? It is late.\n"
    "Agent: I'm sorry for the delay. I can check the latest tracking details for you.\n\n"
    "Customer: My package has not arrived. Can you refund me?\n"
    "Agent:"
)

try:
    from transformers import pipeline, set_seed
    set_seed(42)
    text_generator = pipeline("text-generation", model="distilgpt2", device=-1)  # CPU
    out = text_generator(
        few_shot_prompt,
        max_new_tokens=40,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.2,
        pad_token_id=text_generator.tokenizer.eos_token_id,
        return_full_text=False,
    )
    draft = out[0]["generated_text"].split("\nCustomer:")[0].strip()   # stop at the next turn
    print("Model draft:", draft)
    print("Guardrail flags:", check_reply(draft) or "none")
    print("\nReminder: small demo model. Review and edit every output before use.")
except Exception as e:
    print("Skipped optional model cell:", type(e).__name__, "-", str(e)[:200])

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Model draft: No, it was a mistake in this case because they are already waiting to receive their orders from them (please get your money back)
Guardrail flags: none

Reminder: small demo model. Review and edit every output before use.


### 2D — Evaluation rubric helper *(new)*
Turns the workbook's rubric into something you can score and total.

In [7]:
CRITERIA = ["accuracy", "helpfulness", "tone", "safety", "format"]

def score_output(name, **scores):
    """scores: 0-2 per criterion (0 = fails, 1 = partial, 2 = good)."""
    missing = [c for c in CRITERIA if c not in scores]
    if missing:
        raise ValueError(f"missing criteria: {missing}")
    if any(not 0 <= v <= 2 for v in scores.values()):
        raise ValueError("each score must be 0, 1 or 2")
    total = sum(scores[c] for c in CRITERIA)
    print(f"{name}: {total}/{2*len(CRITERIA)}", scores)
    return total

score_output("Rule-based fallback", accuracy=2, helpfulness=1, tone=2, safety=2, format=2)

Rule-based fallback: 9/10 {'accuracy': 2, 'helpfulness': 1, 'tone': 2, 'safety': 2, 'format': 2}


9

## Part 3 — Image generation
### 3A — Image prompts
**Improvement:** a small builder so learners always specify subject, composition, lighting, style, aspect ratio, and exclusions. Exclusions go in a separate **negative prompt**, because diffusion models handle "no text" inside the main prompt unreliably.

In [8]:
def build_image_prompt(subject, composition, lighting, style, aspect_ratio, exclusions):
    positive = f"{subject}, {composition}, {lighting}, {style}, {aspect_ratio} layout"
    negative = ", ".join(exclusions)
    return positive, negative

image_prompt, negative_prompt = build_image_prompt(
    subject="clean modern e-commerce banner for a sustainable home range, reusable kitchen products on a light wooden counter",
    composition="uncluttered, products arranged with empty space on one side",
    lighting="natural daylight, warm neutral palette",
    style="realistic product photography",
    aspect_ratio="wide 16:9",
    exclusions=["text", "logos", "watermark", "blurry", "clutter"],
)
print("PROMPT:  ", image_prompt)
print("NEGATIVE:", negative_prompt)

# Learner task — sample solution: winter-sale banner
winter_prompt, winter_negative = build_image_prompt(
    subject="winter sale banner for a home-goods store, cozy knitted throws and ceramic mugs on a pale oak table",
    composition="rule-of-thirds, products on the right, empty space on the left",
    lighting="soft window light, cool morning tones with warm accents",
    style="realistic lifestyle photography, shallow depth of field",
    aspect_ratio="wide 16:9",
    exclusions=["text", "logos", "watermark", "people", "snowflake clutter"],
)
print("\nWINTER:  ", winter_prompt)

PROMPT:   clean modern e-commerce banner for a sustainable home range, reusable kitchen products on a light wooden counter, uncluttered, products arranged with empty space on one side, natural daylight, warm neutral palette, realistic product photography, wide 16:9 layout
NEGATIVE: text, logos, watermark, blurry, clutter

WINTER:   winter sale banner for a home-goods store, cozy knitted throws and ceramic mugs on a pale oak table, rule-of-thirds, products on the right, empty space on the left, soft window light, cool morning tones with warm accents, realistic lifestyle photography, shallow depth of field, wide 16:9 layout


### 3B — Optional: Stable Diffusion (needs a GPU)
**Fixes:** skips cleanly (instead of running for hours) on CPU; imports `display`; uses `negative_prompt`; fixed seed; 768×432 (multiples of 8, closer to 16:9 than the default 512×512); attention slicing to lower memory use; clear error message if the model ID is unavailable or gated.
Switch to **Runtime → Change runtime type → T4 GPU** first. Check the current model card and licence before use; model repositories and access terms change.

In [9]:
!pip -q install diffusers transformers accelerate safetensors

In [10]:
from IPython.display import display

MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"   # verify on huggingface.co before class

try:
    import torch
    has_gpu = torch.cuda.is_available()
except ImportError:
    has_gpu = False

if not has_gpu:
    print("No GPU detected - skipping image generation. Use the prompts from 3A instead.")
else:
    try:
        from diffusers import DiffusionPipeline
        pipe = DiffusionPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
        pipe = pipe.to("cuda")
        pipe.enable_attention_slicing()
        generator = torch.Generator("cuda").manual_seed(42)
        image = pipe(
            image_prompt,
            negative_prompt=negative_prompt,
            width=768, height=432,
            num_inference_steps=25,
            guidance_scale=7.0,
            generator=generator,
        ).images[0]
        display(image)
        image.save("brightcart_banner.png")
        print("Saved: brightcart_banner.png")
    except Exception as e:
        print("Image generation failed:", type(e).__name__, "-", str(e)[:300])
        print("Check the model ID/access terms and GPU memory; do not retry blindly.")

No GPU detected - skipping image generation. Use the prompts from 3A instead.


### 3C — Model approach comparison (see workbook table)
No code needed: DALL·E = hosted service; Stable Diffusion = latent diffusion family; GAN = generator vs discriminator; Transformer = architecture.

## Part 4 — Audio (text to speech)
**Fixes:** each cell installs/imports what it needs (4B originally depended on imports from 4A), network errors are caught, and the saved file is verified to exist and be non-empty. Run the install cell first, then restart is *not* required.

In [11]:
!pip -q install gTTS

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wandb 0.28.1 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.
spacy 3.8.16 requires click<9.0.0,>=8.2.1, but you have click 8.1.8 which is incompatible.
huggingface-hub 1.33.0 requires click<9.0.0,>=8.4.2, but you have click 8.1.8 which is incompatible.


In [12]:
import os
from IPython.display import Audio, display

def make_audio(text, filename, lang="en", slow=False):
    """Save text as an MP3 using gTTS. Returns the filename, or None on failure."""
    if not text.strip():
        raise ValueError("text must not be empty")
    try:
        from gtts import gTTS
        gTTS(text=text, lang=lang, slow=slow).save(filename)
    except Exception as e:
        print("Audio generation failed (needs internet + gTTS):", type(e).__name__, "-", str(e)[:200])
        return None
    if not os.path.exists(filename) or os.path.getsize(filename) == 0:
        print("Audio file was not created.")
        return None
    print(f"Saved: {filename} ({os.path.getsize(filename)} bytes)")
    return filename

announcement = (
    "Hello from BrightCart. Your order is delayed. "
    "We are checking the latest tracking details and will share an update when confirmed."
)
assert check_reply(announcement) == []   # same guardrail as text replies (run Cell 2B+ first)
path = make_audio(announcement, "brightcart_update.mp3")
if path:
    display(Audio(path))

Saved: brightcart_update.mp3 (69696 bytes)


In [13]:
# 4B - change the script; compare normal vs slow speed
short = ("Thanks for shopping with BrightCart. "
         "Your order update is available in your account.")
for fname, slow in [("brightcart_short.mp3", False), ("brightcart_short_slow.mp3", True)]:
    p = make_audio(short, fname, slow=slow)
    if p:
        display(Audio(p))

Saved: brightcart_short.mp3 (43968 bytes)


Saved: brightcart_short_slow.mp3 (52992 bytes)


## Part 5 — Code generation
### 5A — Prompt for code (unchanged)

In [14]:
code_prompt = '''
Write a Python function calculate_discount(price, percent).
Requirements:
- price must be numeric and not negative.
- percent must be between 0 and 100 inclusive.
- return the final price after discount, rounded to 2 decimal places.
- raise ValueError for invalid inputs.
- include three example tests.
Return only Python code.
'''
print(code_prompt)


Write a Python function calculate_discount(price, percent).
Requirements:
- price must be numeric and not negative.
- percent must be between 0 and 100 inclusive.
- return the final price after discount, rounded to 2 decimal places.
- raise ValueError for invalid inputs.
- include three example tests.
Return only Python code.



### 5B — Reference solution (hardened) and test suite
**Bugs found in the original reference solution** (all reproduced when testing):
- `calculate_discount(float("nan"), 10)` returned `nan` and `float("inf")` returned `inf` — both should be rejected.
- A `Decimal` price was rejected as "not numeric", even though the workbook recommends `Decimal` for money.
- Only 2 invalid cases were tested, and the `try/except` blocks passed silently if **no** error was raised (a test that cannot fail).

Below: NaN/inf rejection, a `Decimal`-based money version with explicit rounding, and a real test suite where *not raising* is a failure.

In [15]:
import math
from decimal import Decimal, ROUND_HALF_UP, InvalidOperation

def calculate_discount(price, percent):
    """Float version (as in the workbook, hardened)."""
    for name, value in (("price", price), ("percent", percent)):
        if not isinstance(value, (int, float)) or isinstance(value, bool):
            raise ValueError(f"{name} must be numeric")
        if not math.isfinite(value):
            raise ValueError(f"{name} must be a finite number")
    if price < 0:
        raise ValueError("price cannot be negative")
    if not 0 <= percent <= 100:
        raise ValueError("percent must be between 0 and 100")
    return round(price * (1 - percent / 100), 2)


def _to_decimal(name, value):
    if isinstance(value, bool):
        raise ValueError(f"{name} must be numeric")
    try:
        d = Decimal(str(value))
    except (InvalidOperation, ValueError):
        raise ValueError(f"{name} must be numeric")
    if not d.is_finite():
        raise ValueError(f"{name} must be a finite number")
    return d


def calculate_discount_money(price, percent):
    """Decimal version for money: accepts int, float, str or Decimal; rounds half-up."""
    p, pc = _to_decimal("price", price), _to_decimal("percent", percent)
    if p < 0:
        raise ValueError("price cannot be negative")
    if not 0 <= pc <= 100:
        raise ValueError("percent must be between 0 and 100")
    return (p * (Decimal(100) - pc) / Decimal(100)).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)


def run_discount_tests():
    # normal + boundary cases
    assert calculate_discount(100, 10) == 90.0
    assert calculate_discount(249.99, 20) == 199.99
    assert calculate_discount(50, 0) == 50.0
    assert calculate_discount(100, 100) == 0.0
    assert calculate_discount(0, 50) == 0.0
    assert calculate_discount(100, 15) == 85.0           # capstone 15% campaign
    assert calculate_discount(19.99, 15) == 16.99

    # every invalid input MUST raise ValueError (not raising = test failure)
    invalid = [
        (-5, 10), (100, 101), (100, -1), ("100", 10), (100, "10"), (None, 10),
        (True, 10), (100, False), (float("nan"), 10), (10, float("nan")),
        (float("inf"), 10), (10, float("inf")),
    ]
    for args in invalid:
        try:
            calculate_discount(*args)
        except ValueError:
            continue
        raise AssertionError(f"expected ValueError for {args!r}")

    # Decimal money version
    assert calculate_discount_money("249.99", 20) == Decimal("199.99")
    assert calculate_discount_money(Decimal("100.00"), 15) == Decimal("85.00")
    assert calculate_discount_money(1.005, 0) == Decimal("1.01")   # half-up; float round() gives 1.0
    assert calculate_discount_money(100, 100) == Decimal("0.00")
    for args in [(-1, 10), (100, 101), ("abc", 10), (True, 10), ("NaN", 10), (10, "Infinity")]:
        try:
            calculate_discount_money(*args)
        except ValueError:
            continue
        raise AssertionError(f"expected ValueError for {args!r}")
    print("All discount tests passed.")

run_discount_tests()
print("Float vs Decimal on 1.005 with 0% off:", calculate_discount(1.005, 0), "vs", calculate_discount_money(1.005, 0))

All discount tests passed.
Float vs Decimal on 1.005 with 0% off: 1.0 vs 1.01


### 5C — Review checklist for generated code
- Does it satisfy every requirement (including NaN/inf, booleans, strings)?
- Does the test suite fail when the code is wrong? (Try breaking the function on purpose.)
- Are errors clear and specific?
- Does it leak secrets, call unknown services, or perform destructive actions?
- Can another developer read and maintain it?

## Part 7 — BrightCart capstone (one runnable cell)
Runs the text, image prompt, audio and code steps, then prints a self-check. Needs the cells above to have been run (functions are reused).

In [16]:
# TEXT
customer_reply = brightcart_fallback("My package has not arrived. Can you refund me?", "delayed")
text_flags = check_reply(customer_reply)

# IMAGE PROMPT (+ generated image from 3B if a GPU was available)
campaign_prompt, campaign_negative = build_image_prompt(
    subject="reusable kitchen products on a light wooden counter",
    composition="uncluttered, space for copy added later in design software",
    lighting="natural daylight, warm neutral palette",
    style="realistic product photography",
    aspect_ratio="wide 16:9",
    exclusions=["text", "logos", "watermark"],
)

# AUDIO
audio_path = make_audio(customer_reply, "capstone_update.mp3")

# CODE
campaign_price = calculate_discount(100, 15)
campaign_price_money = calculate_discount_money("100.00", 15)

print("\n=== CAPSTONE SELF-CHECK ===")
print("Text reply        :", customer_reply)
print("Guardrail flags   :", text_flags or "none")
print("Image prompt      :", campaign_prompt)
print("Negative prompt   :", campaign_negative)
print("Audio file        :", audio_path or "not generated (no internet / gTTS)")
print("15% on 100        :", campaign_price, "|", campaign_price_money)
assert text_flags == [] and campaign_price == 85.0 and campaign_price_money == Decimal("85.00")
print("Capstone checks passed.")

Saved: capstone_update.mp3 (73728 bytes)

=== CAPSTONE SELF-CHECK ===
Text reply        : I'm sorry your order is delayed. I can help check the latest tracking details. Your refund eligibility has not yet been confirmed.
Guardrail flags   : none
Image prompt      : reusable kitchen products on a light wooden counter, uncluttered, space for copy added later in design software, natural daylight, warm neutral palette, realistic product photography, wide 16:9 layout
Negative prompt   : text, logos, watermark
Audio file        : capstone_update.mp3
15% on 100        : 85.0 | 85.00
Capstone checks passed.


## Part 8 — Knowledge check (answers in the workbook)
Try answering aloud before checking: rule-based vs LLM · Transformer as architecture · diffusion vs GAN · why DALL·E needs provider access · why test AI code · handling unknown refund eligibility · what gTTS does · three image-prompt constraints.